In [4]:
!pip install pandas numpy scikit-learn imbalanced-learn xgboost lightgbm tensorflow 

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
import os
import gc
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

3. Configuracion general

In [3]:
@dataclass
class Config:
    input_file: str = "transformed_it_ot.csv"
    label_col: str = "traffic_label"
    drop_cols: tuple = ("line_number", "attack_type")
    test_size: float = 0.20
    random_state: int = 42
    max_rows_for_full_train: int = 2500000
    k_best: int = 0
    smote_k_neighbors: int = 5
    dnn_epochs: int = 20
    dnn_batch_size: int = 1024
    dnn_patience: int = 5
    n_splits: int = 3

cfg = Config()
print(cfg)

Config(input_file='transformed_it_ot.csv', label_col='traffic_label', drop_cols=('line_number', 'attack_type'), test_size=0.2, random_state=42, max_rows_for_full_train=2500000, k_best=0, smote_k_neighbors=5, dnn_epochs=20, dnn_batch_size=1024, dnn_patience=5, n_splits=3)


4. Funciones auxiliares

In [4]:
def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = downcast_numeric(df)
    return df

def gc_clean(*objs):
    for obj in objs:
        del obj
    gc.collect()

5. Carga y limpieza inicial

In [5]:
print(cfg.input_file)

transformed_it_ot.csv


In [6]:
print("Cargando dataset...")
df = pd.read_csv(cfg.input_file, low_memory=False, encoding="utf-8-sig")

# Normalizar nombres de columnas
df.columns = (
    df.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
    .str.lower()
)

print("Columnas detectadas:")
print(df.columns.tolist())

leakage_cols = [c for c in cfg.drop_cols if c in df.columns]
if leakage_cols:
    df = df.drop(columns=leakage_cols)
    print("Columnas eliminadas por leakage:", leakage_cols)

# Buscar la columna objetivo de forma robusta
posibles_labels = ["traffic_label", "label", "attack_label", "target", "class"]
col_objetivo = None
for c in posibles_labels:
    if c in df.columns:
        col_objetivo = c
        break

if col_objetivo is None:
    raise KeyError(
        "No se encontro una columna objetivo valida. "
        "Se esperaba traffic_label o un nombre equivalente."
    )

cfg.label_col = col_objetivo
print("Columna objetivo detectada:", cfg.label_col)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[cfg.label_col]).reset_index(drop=True)
df[cfg.label_col] = df[cfg.label_col].astype("int32")

df = reduce_memory(df)

print("Forma despues de limpieza:", df.shape)
print("Memoria aproximada (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
gc.collect()

df.head()

Cargando dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'transformed_it_ot.csv'

6. Submuestreo estratificado (Si la RAM lo requiere)

7. Separacion train/test

In [10]:
# Revalidacion estricta antes del split
print("Columnas actuales en df:")
print(df.columns.tolist())

# Normalizar otra vez por seguridad
df.columns = df.columns.astype(str).str.strip().str.lower()

# Si la etiqueta quedo como indice por algun cambio previo, la recupero
if cfg.label_col not in df.columns:
    if cfg.label_col in df.index.names:
        df = df.reset_index()
        df.columns = df.columns.astype(str).str.strip().str.lower()

# Detectar etiqueta de forma segura
posibles_labels = ["traffic_label", "label", "attack_label", "target", "class"]
col_objetivo = None

for c in posibles_labels:
    if c in df.columns:
        col_objetivo = c
        break

if col_objetivo is None:
    raise KeyError(
        f"No se encontro una columna objetivo valida. Columnas disponibles: {df.columns.tolist()}"
    )

cfg.label_col = col_objetivo
print("Columna objetivo final:", cfg.label_col)

y = df[cfg.label_col].astype("int32").values
X = df.drop(columns=[cfg.label_col])

feature_names = X.columns.tolist()

print("Numero de features:", len(feature_names))
print("Distribucion de clases:")
print(pd.Series(y).value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y,
    test_size=cfg.test_size,
    random_state=cfg.random_state,
    stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
gc_clean(df, X, y)

Columnas actuales en df:
['src_port', 'dst_port', 'protocol', 'duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_packets', 'dst2src_packets', 'src2dst_bytes', 'dst2src_bytes', 'src2dst_duration_ms', 'dst2src_duration_ms', 'application_name', 'application_category', 'traffic_label', 'log_length', 'device_type', 'hour', 'minute', 'second']
Columna objetivo final: traffic_label
Numero de features: 19
Distribucion de clases:
0    3822077
1      85459
Name: count, dtype: int64
X_train: (3126028, 19) X_test: (781508, 19)


8. Escalado

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.astype("float32"))
X_test_scaled = scaler.transform(X_test.astype("float32"))

X_train_scaled = X_train_scaled.astype("float32")
X_test_scaled = X_test_scaled.astype("float32")

print("Escalado completado.")
gc.collect()

Escalado completado.


7

9. Optimizacion de caracteristicas

In [12]:
# Para este volumen y RAM objetivo, uso SelectKBest como alternativa practica a EFA.
# Se ajusta solo con X_train escalado y se transforma train/test sin fuga de datos.

n_features = X_train_scaled.shape[1]
if cfg.k_best <= 0:
    k_best = min(10, n_features)
else:
    k_best = min(cfg.k_best, n_features)

selector = SelectKBest(score_func=mutual_info_classif, k=k_best)
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)

X_train_fs = X_train_fs.astype("float32")
X_test_fs = X_test_fs.astype("float32")

selected_features = [feature_names[i] for i in selector.get_support(indices=True)]
print("Features seleccionadas:", selected_features)
print("Shape train:", X_train_fs.shape, "Shape test:", X_test_fs.shape)
gc.collect()

Features seleccionadas: ['dst_port', 'protocol', 'bidirectional_bytes', 'src2dst_bytes', 'src2dst_duration_ms', 'dst2src_duration_ms', 'application_name', 'application_category', 'device_type', 'hour']
Shape train: (3126028, 10) Shape test: (781508, 10)


345

10. SMOTE solo sobre train

In [ ]:
minority_count = int(np.min(np.bincount(y_train)))
k_neighbors = min(cfg.smote_k_neighbors, max(1, minority_count - 1))

smote = SMOTE(
    sampling_strategy=auto,
    random_state=cfg.random_state,
    k_neighbors=k_neighbors
)

X_train_bal, y_train_bal = smote.fit_resample(X_train_fs, y_train)

X_train_bal = X_train_bal.astype("float32")
y_train_bal = y_train_bal.astype("int32")

print("Despues de SMOTE:", X_train_bal.shape, np.bincount(y_train_bal))
gc.collect()

Despues de SMOTE: (4586491, 10) [3057661 1528830]


23

11. Modelos base

In [14]:
def build_dnn(input_dim: int) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.15),
        Dense(16, activation="relu"),
        Dropout(0.10),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

def build_xgb(y_ref):
    pos = max(1, int(np.sum(y_ref == 1)))
    neg = max(1, int(np.sum(y_ref == 0)))
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=180,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=1,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        scale_pos_weight=neg / pos
    )

def build_lgbm(y_ref):
    pos = max(1, int(np.sum(y_ref == 1)))
    neg = max(1, int(np.sum(y_ref == 0)))
    return LGBMClassifier(
        objective="binary",
        n_estimators=220,
        learning_rate=0.04,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight=neg / pos,
        verbosity=-1
    )

12. Funcion para OOF sin leakage interno

In [15]:
def fit_predict_fold(X_tr, y_tr, X_va, y_va):
    fold_scaler = StandardScaler()
    X_tr_s = fold_scaler.fit_transform(X_tr.astype("float32"))
    X_va_s = fold_scaler.transform(X_va.astype("float32"))

    fold_selector = SelectKBest(score_func=mutual_info_classif, k=min(k_best, X_tr_s.shape[1]))
    X_tr_f = fold_selector.fit_transform(X_tr_s, y_tr)
    X_va_f = fold_selector.transform(X_va_s)

    minority_count = int(np.min(np.bincount(y_tr)))
    k_neighbors = min(cfg.smote_k_neighbors, max(1, minority_count - 1))

    fold_smote = SMOTE(
        sampling_strategy="auto",
        random_state=cfg.random_state,
        k_neighbors=k_neighbors
    )
    X_tr_b, y_tr_b = fold_smote.fit_resample(X_tr_f, y_tr)

    xgb = build_xgb(y_tr_b)
    lgbm = build_lgbm(y_tr_b)
    dnn = build_dnn(X_tr_b.shape[1])

    xgb.fit(X_tr_b, y_tr_b)
    lgbm.fit(X_tr_b, y_tr_b)

    es = EarlyStopping(monitor="val_loss", patience=cfg.dnn_patience, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, cfg.dnn_patience // 2),
        min_lr=1e-6
    )

    dnn.fit(
        X_tr_b, y_tr_b,
        validation_data=(X_va_f, y_va),
        epochs=cfg.dnn_epochs,
        batch_size=cfg.dnn_batch_size,
        verbose=0,
        callbacks=[es, rlrop]
    )

    p_xgb = xgb.predict_proba(X_va_f)[:, 1]
    p_lgbm = lgbm.predict_proba(X_va_f)[:, 1]
    p_dnn = dnn.predict(X_va_f, verbose=0).reshape(-1)

    return np.column_stack([p_xgb, p_lgbm, p_dnn]).astype("float32")

13. Entrenamiento OOF del meta-learner

In [16]:
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_state)
oof_meta = np.zeros((X_train_bal.shape[0], 3), dtype="float32")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_bal, y_train_bal), start=1):
    X_tr, X_va = X_train_bal[tr_idx], X_train_bal[va_idx]
    y_tr, y_va = y_train_bal[tr_idx], y_train_bal[va_idx]

    oof_meta[va_idx] = fit_predict_fold(X_tr, y_tr, X_va, y_va)
    print(f"Fold {fold} completado")

meta_learner = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
meta_learner.fit(oof_meta, y_train_bal)

print("Meta learner entrenado.")
gc.collect()

Fold 1 completado
Fold 2 completado
Fold 3 completado
Meta learner entrenado.


6848

14. Entrenamiento final de base learners

In [17]:
final_xgb = build_xgb(y_train_bal)
final_lgbm = build_lgbm(y_train_bal)
final_dnn = build_dnn(X_train_bal.shape[1])

final_xgb.fit(X_train_bal, y_train_bal)
final_lgbm.fit(X_train_bal, y_train_bal)

es = EarlyStopping(monitor="val_loss", patience=cfg.dnn_patience, restore_best_weights=True)
rlrop = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=max(2, cfg.dnn_patience // 2),
    min_lr=1e-6
)

final_dnn.fit(
    X_train_bal, y_train_bal,
    validation_split=0.15,
    epochs=cfg.dnn_epochs,
    batch_size=cfg.dnn_batch_size,
    verbose=0,
    callbacks=[es, rlrop]
)

print("Modelos base finales entrenados.")
gc.collect()

Modelos base finales entrenados.


1371

15. Evaluacion final en X_test

In [18]:
test_meta = np.column_stack([
    final_xgb.predict_proba(X_test_fs)[:, 1],
    final_lgbm.predict_proba(X_test_fs)[:, 1],
    final_dnn.predict(X_test_fs, verbose=0).reshape(-1)
]).astype("float32")

y_prob = meta_learner.predict_proba(test_meta)[:, 1]
y_pred = (y_prob >= 0.5).astype("int32")

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy :", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall   :", round(rec, 4))
print("F1-Score :", round(f1, 4))
print("\nMatriz de confusion:")
print(cm)
print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Accuracy : 0.9993
Precision: 0.9696
Recall   : 1.0
F1-Score : 0.9846

Matriz de confusion:
[[763881    535]
 [     0  17092]]

Reporte de clasificacion:
              precision    recall  f1-score   support

           0     1.0000    0.9993    0.9996    764416
           1     0.9696    1.0000    0.9846     17092

    accuracy                         0.9993    781508
   macro avg     0.9848    0.9997    0.9921    781508
weighted avg     0.9993    0.9993    0.9993    781508



16. Resultado

In [19]:
results = pd.DataFrame([{
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1_score": f1
}])

results.to_csv("stacked_ensemble_results.csv", index=False)
print("Resultados guardados en stacked_ensemble_results.csv")

Resultados guardados en stacked_ensemble_results.csv
